# Voice Quality Features (IEMOCAP)

This notebook extracts voice-quality features (HNR via parselmouth, or HPSS proxy).
Each utterance becomes one training row for downstream SER models.

In [1]:
from pathlib import Path
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

import librosa
import numpy as np
import pandas as pd


In [2]:
# Configuration
REPO_ROOT = Path.cwd().parents[1]  # repo root (notebook is under feature_extraction/)
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "voice_quality"
OUT_FILE = "voice_quality_features.csv"

# Audio + feature params
TARGET_SR = 16_000
FMIN = 60.0
FMAX = 400.0

EXCLUDED_EMOTIONS = {"sur", "fea", "oth", "dis"}

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


PosixPath('/home/beka/Speech-Emotion-Recognition/extracted_features/voice_quality/voice_quality_features.csv')

In [3]:
def load_audio(path: Path) -> tuple[np.ndarray, int]:
    # Load audio and resample to TARGET_SR so features are comparable
    audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    return audio, sr


def _hnr_from_parselmouth(audio: np.ndarray, sr: int, *, fmin: float) -> float | None:
    # Compute HNR via Praat/parselmouth, returning None if unavailable
    try:
        import parselmouth
        from parselmouth.praat import call
    except ImportError:
        return None

    sound = parselmouth.Sound(audio, sr)
    harmonicity = call(sound, "To Harmonicity (cc)", 0.01, fmin, 0.1, 1.0)
    hnr = call(harmonicity, "Get mean", 0, 0)
    if isinstance(hnr, float) and np.isfinite(hnr):
        return float(hnr)
    return float("nan")


def _hnr_proxy_from_hpss(audio: np.ndarray) -> float:
    # Compute a proxy HNR using harmonic/percussive energy ratio
    harmonic, percussive = librosa.effects.hpss(audio)
    harm_energy = float(np.mean(harmonic ** 2))
    noise_energy = float(np.mean(percussive ** 2))
    total_energy = harm_energy + noise_energy
    if total_energy <= 0.0:
        return float("nan")
    eps = 1e-12
    return float(10.0 * np.log10((harm_energy + eps) / (noise_energy + eps)))


def extract_voice_quality(audio: np.ndarray, sr: int) -> dict[str, float]:
    _ = FMAX  # reserved for future jitter/shimmer extraction
    audio = np.asarray(audio, dtype=np.float32)

    hnr = _hnr_from_parselmouth(audio, sr, fmin=FMIN)
    if hnr is not None:
        return {"vq_hnr_db": float(hnr)}

    hnr_proxy = _hnr_proxy_from_hpss(audio)
    return {"vq_hnr_proxy_db": float(hnr_proxy)}


In [4]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: keep xxx, exclude selected classes, and enforce agreement for labeled classes.
# This keeps unlabeled (xxx) examples while dropping sur/fea/oth/dis.
df = df[~df["emotion"].isin(EXCLUDED_EMOTIONS)].copy()
df = df[(df["emotion"] == "xxx") | (df["agreement"] > 0)].copy()
df.shape


(9887, 7)

In [5]:
CPU_COUNT = os.cpu_count() or 1
COMPUTE_DEVICE = "cpu"  # Force CPU for this CPU-bound extractor
NUM_WORKERS = max(1, CPU_COUNT - 2)
PROGRESS_MIN_INTERVAL = 1.0

print(f"Compute device: {COMPUTE_DEVICE} | extractor_backend=cpu | workers={NUM_WORKERS}")


def process_row(row: dict[str, object]) -> tuple[dict[str, float | str | int] | None, str | None]:
    rel_path = str(row["path"])
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        return None, str(audio_path)

    audio, sr = load_audio(audio_path)
    duration_s = audio.shape[0] / sr
    features = extract_voice_quality(audio, sr)

    record: dict[str, float | str | int] = {
        "path": rel_path,
        "session": int(row["session"]),
        "method": str(row["method"]),
        "gender": str(row["gender"]),
        "emotion": str(row["emotion"]),
        "n_annotators": int(row["n_annotators"]),
        "agreement": int(row["agreement"]),
        "duration_s": float(duration_s),
    }
    record.update(features)
    return record, None


rows: list[dict[str, float | str | int]] = []
missing: list[str] = []
records = df.to_dict(orient="records")

if NUM_WORKERS > 1:
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        mapped = executor.map(process_row, records)
        for record, missing_path in tqdm(mapped, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
            if missing_path is not None:
                missing.append(missing_path)
                continue
            if record is not None:
                rows.append(record)
else:
    for record in tqdm(records, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
        row_result, missing_path = process_row(record)
        if missing_path is not None:
            missing.append(missing_path)
            continue
        if row_result is not None:
            rows.append(row_result)

feature_df = pd.DataFrame(rows)
feature_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
print(f"Workers used: {NUM_WORKERS} (cpu_count={CPU_COUNT})")
if missing:
    print(f"Missing audio files: {len(missing)}")
feature_df.shape


Compute device: cpu | extractor_backend=cpu | workers=62


Extracting:   0%|          | 0/9887 [00:00<?, ?file/s]

Saved: /home/beka/Speech-Emotion-Recognition/extracted_features/voice_quality/voice_quality_features.csv
Workers used: 62 (cpu_count=64)


(9887, 9)